In [1]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

In [2]:
drive.mount('/content/drive')

CLEANED_DIR = '/content/drive/MyDrive/My Project/E-Commerce Data Modeling & Customer Lifetime Value Analytics/data/cleaned'
STAR_SCHEMA_DIR = '/content/drive/MyDrive/My Project/E-Commerce Data Modeling & Customer Lifetime Value Analytics/data/star_schema'

Mounted at /content/drive


In [3]:
df_orders = pd.read_csv(f'{CLEANED_DIR}/cleaned_orders.csv')
df_customers = pd.read_csv(f'{CLEANED_DIR}/cleaned_customers.csv')
df_items = pd.read_csv(f'{CLEANED_DIR}/cleaned_items.csv')
df_products = pd.read_csv(f'{CLEANED_DIR}/cleaned_products.csv')
df_sellers = pd.read_csv(f'{CLEANED_DIR}/cleaned_sellers.csv')
df_payments = pd.read_csv(f'{CLEANED_DIR}/cleaned_payments.csv')
df_reviews = pd.read_csv(f'{CLEANED_DIR}/cleaned_reviews.csv')

print("Xong")

Xong


In [4]:
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    df_orders[col] = pd.to_datetime(df_orders[col])

In [5]:
# DIM_CUSTOMER
Dim_Customer = df_customers[['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']].copy()
Dim_Customer['customer_segment'] = 'Unassigned'

Dim_Customer.head(5)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_segment
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,Unassigned
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,Unassigned
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,Unassigned
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,Unassigned
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,Unassigned


In [6]:
# DIM_PRODUCT
Dim_Product = df_products[['product_id', 'product_category_name', 'product_weight_g',
                           'product_length_cm', 'product_height_cm', 'product_width_cm']].copy()
Dim_Product['product_volume_cm3'] = Dim_Product['product_length_cm'] * Dim_Product['product_height_cm'] * Dim_Product['product_width_cm']

Dim_Product.head(5)

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_volume_cm3
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,225.0,16.0,10.0,14.0,2240.0
1,3aa071139cb16b67ca9e5dea641aaa2f,art,1000.0,30.0,18.0,20.0,10800.0
2,96bd76ec8810374ed1b65e291975717f,sports_leisure,154.0,18.0,9.0,15.0,2430.0
3,cef67bcfe19066a932b7673e239eb23d,baby,371.0,26.0,4.0,26.0,2704.0
4,9dc1a7de274444849c219cff195d0b71,housewares,625.0,20.0,17.0,13.0,4420.0


In [7]:
# DIM_SELLER
Dim_Seller = df_sellers[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']].copy()

Dim_Seller.head(5)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [8]:
# DIM_DATE
start_date = df_orders['order_purchase_timestamp'].min().normalize()
end_date = df_orders['order_purchase_timestamp'].max().normalize()
date_range = pd.date_range(start=start_date, end=end_date)

Dim_Date = pd.DataFrame({'date': date_range})
Dim_Date['date_key'] = Dim_Date['date'].dt.strftime('%Y%m%d').astype(int)
Dim_Date['year'] = Dim_Date['date'].dt.year
Dim_Date['month'] = Dim_Date['date'].dt.month
Dim_Date['quarter'] = Dim_Date['date'].dt.quarter
Dim_Date['day_of_week'] = Dim_Date['date'].dt.day_name()
Dim_Date['is_weekend'] = Dim_Date['date'].dt.dayofweek.isin([5, 6]).astype(int)

Dim_Date.head(5)

,date,date_key,year,month,quarter,day_of_week,is_weekend
0,2016-09-04,20160904,2016,9,3,Sunday,1
1,2016-09-05,20160905,2016,9,3,Monday,0
2,2016-09-06,20160906,2016,9,3,Tuesday,0
3,2016-09-07,20160907,2016,9,3,Wednesday,0
4,2016-09-08,20160908,2016,9,3,Thursday,0


In [10]:
# DIM_PAYMENT
Dim_Payment = df_payments[['payment_type', 'payment_installments']].drop_duplicates().reset_index(drop=True)
Dim_Payment['payment_key'] = Dim_Payment.index + 1

Dim_Payment.head(5)

,payment_type,payment_installments,payment_key
0,credit_card,8,1
1,credit_card,1,2
2,credit_card,2,3
3,credit_card,3,4
4,credit_card,6,5


In [11]:
Fact_Orders = df_items.merge(df_orders[['order_id', 'customer_id', 'order_purchase_timestamp',
                                        'order_delivered_customer_date', 'order_estimated_delivery_date']],
                             on='order_id', how='inner')

# Tạo order_date_key để join với Dim_Date
Fact_Orders['order_date_key'] = Fact_Orders['order_purchase_timestamp'].dt.strftime('%Y%m%d').astype(int)

# Tính toán các metric giao hàng
Fact_Orders['delivery_days'] = (Fact_Orders['order_delivered_customer_date'] - Fact_Orders['order_purchase_timestamp']).dt.days
Fact_Orders['is_late_delivery'] = (Fact_Orders['order_delivered_customer_date'] > Fact_Orders['order_estimated_delivery_date']).astype(int)

Fact_Orders = Fact_Orders.merge(df_reviews[['order_id', 'review_score']], on='order_id', how='left')

main_payment = df_payments.sort_values(['order_id', 'payment_value'], ascending=[True, False]).drop_duplicates('order_id')
main_payment = main_payment.merge(Dim_Payment, on=['payment_type', 'payment_installments'], how='left')
Fact_Orders = Fact_Orders.merge(main_payment[['order_id', 'payment_key']], on='order_id', how='left')

# Các cột cần thiết cho Fact Table
fact_cols = ['order_id', 'order_item_id', 'customer_id', 'product_id', 'seller_id',
             'order_date_key', 'payment_key', 'price', 'freight_value',
             'review_score', 'delivery_days', 'is_late_delivery']

Fact_Orders = Fact_Orders[fact_cols].copy()
Fact_Orders.rename(columns={'price': 'payment_value'}, inplace=True)

In [12]:
Fact_Orders.head(5)

,order_id,order_item_id,customer_id,product_id,seller_id,order_date_key,payment_key,payment_value,freight_value,review_score,delivery_days,is_late_delivery
0,00010242fe8c5a6d1ba2dd792cb16214,1,3ce436f183e68e07877b285a838db11a,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,20170913,3.0,58.90,13.29,5.0,7.0,0
1,00018f77f2f0320c557190d7a144bdd3,1,f6dd3ec061db4e3987629fe6b26e5cce,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,20170426,4.0,239.90,19.93,4.0,16.0,0
2,000229ec398224ef6ca0657da4fc703e,1,6489ae5e4333f3693df5ad4372dab6d3,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,20180114,7.0,199.00,17.87,5.0,7.0,0
3,00024acbcdf0a6daa1e931b038114c75,1,d4eb9395c8c0431ee92fce09860c5a06,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,20180808,3.0,12.99,12.79,4.0,6.0,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,58dbd0b2d70206bf40e62cd34e84d795,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,20170204,4.0,199.90,18.14,5.0,25.0,0


In [14]:
Fact_Orders.to_csv(f'{STAR_SCHEMA_DIR}/Fact_Orders.csv', index=False)
Dim_Customer.to_csv(f'{STAR_SCHEMA_DIR}/Dim_Customer.csv', index=False)
Dim_Product.to_csv(f'{STAR_SCHEMA_DIR}/Dim_Product.csv', index=False)
Dim_Seller.to_csv(f'{STAR_SCHEMA_DIR}/Dim_Seller.csv', index=False)
Dim_Date.to_csv(f'{STAR_SCHEMA_DIR}/Dim_Date.csv', index=False)
Dim_Payment.to_csv(f'{STAR_SCHEMA_DIR}/Dim_Payment.csv', index=False)

print("Xong")

Xong
